In [1]:
import random
import copy
import numbers
import math
from sympy import *
from itertools import combinations

In [ ]:
def back_substitute(d,k,v):
    """Substitutes k-->v in the values of d
    INPUTS:
    * 'd' - a dictionary
    * 'k' - a symbol for substitution
    * 'v' - a value for substitution"""
    for A in d:
        d[A]=d[A].subs(k,v)

In [ ]:
def Indexed_obj_in_expr(expr):
    """Returns the set of Indexed obejcts which appear in expr
    INPUTS:
    * 'expr' - an algebraic expression in Indexed objects and symbols"""
    r=set()
    if isinstance(expr,numbers.Number): return r
    if type(expr)==Add or type(expr)==Mul:
        for A in expr.args:
            r=r.union(Indexed_obj_in_expr(A))
    if type(expr)==Pow:
        r=r.union(Indexed_obj_in_expr(expr.base))
        r=r.union(Indexed_obj_in_expr(expr.exp))
    if type(expr)==Indexed:
        r.add(expr)
    if type(expr)==Symbol: return r
    if type(expr)==exp:
        for A in expr.args:
            r=r.union(Indexed_obj_in_expr(A))
    return r

In [ ]:
def check_if_subs_are_needed(d):
    for A in d:
        if Indexed_obj_in_expr(d[A]).intersection(set(d.keys()))!=set():
            print('Substitutions needed')
            return
    print('No subsitutions needed')

In [117]:
def ind_diag(*elts):
    """
    this is a version of diag with accepts Indexed objects
    """
    r=zeros(len(elts))
    for i in range(len(elts)):
        r[i,i]=elts[i]
    return r

In [2]:
def colspace_containment(M,N):
    """Checks if the columnspace of M is contained in the columnspace of N
    INPUTS:
    * 'M', 'N' - Matrices
    """
    N_cols=[list(N.col(i)) for i in N.rref()[1]]
    col_list=N_cols+[list(M.col(i)) for i in range(shape(M)[1])]
    return len(Matrix(col_list).rref()[1])<=len(N_cols)

In [3]:
def map_to_vd(F,vd):
    """Applies the function F to each vector from vd, returning a new vd
    INPUTS:
    * 'F' - a function
    * 'vd' - a vector dictionary
    """

In [4]:
def Mat_adjoint(A,Q_dom,Q_codom):
    """Returns the adjoint matrix of A with respect to the inner products
    defined by Q_dom and Q_codom
    INPUTS:
    * 'A' - A matrix representing a linear transformation from dom-->codom
    * 'Q_dom' - A matrix representing a 2-form on dom
    * 'Q_codom' - A matrix representing a 2-form on codom
    """
    if A==Matrix([[]]): return Matrix([[]])
    return (Q_codom*A*Q_dom.inv()).transpose()

In [5]:
def ortho_proj(v, basis, Q):
    """Gives the orthogonal projection with respect to Q of 
    the vector v onto the space spanned by basis
    INPUTS:
    * 'v' - a list or vector
    * 'basis' - a matrix with linearly independent columns of length len(v)
    * 'Q' - a len(v)xlen(v) matrix, representing a 2-form
    """
    # Convert v and basis to SymPy matrices (column vectors)
    u = SparseMatrix(v)
    if shape(u)[1]!=1: u=u.transpose()
    
    A = SparseMatrix(basis)
    if shape(A)[1]==0: return zeros(*shape(u))
    
    # Compute Gramian matrix G
    G = A.transpose() * Q * A

    # Solve linear system G * x = A.T * Q * v
    x = G.solve(A.transpose() * Q * u)
    
    # Compute projection
    projection = A * x
    
    return projection

In [6]:
def update_add_vd(vd,d,w,i,c):
    """Adds c times the specified basis element to self
    INPUTS:
    * 'vd' = vector dictionary
    * 'd' = degree
    * 'w' = weight
    * 'i' = index
    * 'c' = coefficient
    """
    if not d in vd: self.vd[d]={}
    if not w in vd[d]: self.vd[d][w]=SparseMatrix(zeros(len(self.parent.basis[d][w]),1))
    self.vd[d][w][i]=self.vd[d][w][i]+c
    
def update_add_mat(Md,d,w,M):
    """Adds vector v to the specified degree and weight of self
    INPUTS:
    * 'Md' - Matrix dictionary
    * 'd' - degree
    * 'w' - weight
    * 'M' - Matrix
    """
    if not d in Md: Md[d]={}
    if not w in Md[d]: Md[d][w]=SparseMatrix(M)
    else: Md[d][w]=SparseMatrix(Md[d][w]+M)

def update_add_dict(Md1,Md2,coeff=1):
    """Adds the vector dict coeff*Md2 to Md1, updating Md1
    INPUTS:
    * 'Md1','Md2' - matrix dictionaries
    * 'coeff' - a coefficient
    """
    if coeff==0: return None
    for d in Md2:
        for w in Md2[d]: update_add_mat(Md1,d,w,coeff*Md2[d][w])

In [7]:

# v0=SparseMatrix(zeros(3,1))
# v1=SparseMatrix([1,0,0])
# v2=SparseMatrix([1,1,1])
# M0=SparseMatrix(eye(3))
# M1=SparseMatrix([[2,0,0],[0,0,0],[0,0,0]])

# tvd0={}
# tvd1={0:{0:v0}}
# tvd2={0:{0:v1}}
# tvd3={0:{1:v2}}
# tvd4={1:{0:v1}}

# tMd1={0:{0:M0}}
# tMd2={0:{1:M0,2:M1},1:{2:M1}}
# tMd3={0:{1:M1},1:{1:M0}}

In [8]:
def wedge_vecs(vec_list):
    """Returns a vector representing the wedge product of the given vectors
    """
    if len(vec_list)==0: return None
    M=Matrix(list(map(list,vec_list)))
    result=zeros(binomial(len(vec_list[0]),len(vec_list)),1)
    L=list(combinations(range(len(vec_list[0])),len(vec_list)))
    for j in range(len(L)):
        result[j]+=SparseMatrix([list(M.col(k)) for k in L[j]]).det()
    return result

In [9]:
def cd_to_vec(cd,base,base_index_dict):
    v=SparseMatrix([0]*len(base))
    print(v)
    for k in cd:
        v[base_index_dict[k]]=cd[k]
    return v

In [10]:
def remove_zeros_vd(vd):
    # This could be quite costly if the equality checks are complicated...
    vdk=list(vd.keys())
    for d in vdk:
        vddk=list(vd[d].keys())
        for w in vddk:
            if vd[d][w]==zeros(*shape(vd[d][w])): vd[d].pop(w)
        if vd[d]=={}: vd.pop(d)

In [11]:
def hrs_min_sec(sec_val):
    hours=str(sec_val//(60**2))
    minutes=str((sec_val//60)%60)
    seconds=str(round(sec_val%60,0))
    if sec_val//(60**2)!=0:
        return(hours+' hrs '+minutes+' min '+seconds+' sec')
    if (sec_val//60)%60!=0:
        return(minutes+' min '+seconds+' sec')
    return(seconds+' sec')

In [12]:
def str_from_vd(vd,b):
    """ Produces a string from a vector dictionary
    INPUTS:
    * vd - a vector dictionary; a dict of dicts of vectors with keys deg then wght
    * b - a basis method returning a dict of dicts of basis elts, with keys deg then wght
    """
    r=''
    for d in vd:
        for w in vd[d]:
            for i in range(len(vd[d][w])):
                r=append_monomial(r,vd[d][w][i],b(d,w)[i])
    if r=='': return '0'
    return r

In [13]:
def append_monomial(s,c,p):
    """Returns a new string consisting of s + c*p
    INPUTS:
    * 's' - a string
    * 'c' - a (numeric or symbolic) coefficient
    * 'p' - a string or basis element
    """
    m=mul_str(c,p)
    if m=='':return s
    if s=='': return m
    if m[0] in ['-','+']: return s+m
    return s+'+'+m
    
def mul_str(c,p):
    """Returns a string consisting of c*p
    INPUTS:
    * 'c' - a (numeric or symbolic) coefficient
    * 'p' - a string or basis element
    """
    if c==0: return ''
    if c==-1: return '-'+str(p)
    if c==1: return str(p)
    if type(c)==Add: return '('+str(c)+')*'+str(p)
    return str(c)+'*'+str(p)

In [14]:
def sort_basis_tuple(basis_tuple,basis):
    '''Returns: a tuple containing an rearrangement of basis_tuple of descending degree, 
    and the sign of the permutation (either -1 or 1)
    
    INPUTS:
    * basis: a list of elements
    * basis_tuple: a tuple or list of elts from basis'''
    
    basis_list=list(basis_tuple)
    sorted_list=basis_list.copy()
    sorted_list.sort(key=lambda A:basis.index(A))
    return(tuple(sorted_list),permutation_sign(basis_list,sorted_list))

In [15]:
def find_cochain_basis(ss):
    '''args: ss (spanning set), a list of cochains of the same homogeneous degree
       Returns: A list of cochains which are a basis for the subspace spanned by ss'''
    if len(ss)==0: return []
    basis_set=set()
    for c in ss:
        for base_elt in c.coeff_dict:
            basis_set.add(base_elt)
    B=list(basis_set)
    
    M=zeros(len(ss),len(B))
    for i in range(len(ss)):
        set_row(M,i,coordinatize_cochain_in_basis(ss[i],B))
    M=M.rref()[0]
    result=[list(M.row(i)) for i in range(shape(M)[0]) 
            if list(M.row(i))!=[0]*len(M.row(i))]   
    basis_cochains=[cochain({A:1},ss[0].parent) for A in B]
    return([coords_to_lin_comb(A,basis_cochains) for A in result])

In [1]:
def simplify_cochain(c):
    for d in c.vd:
        for w in c.vd[d]:
            c.vd[d][w]=simplify(c.vd[d][w])

In [17]:
def set_row(mat,rowNum,row):
    if type(row)==type(zeros(3,3)):
        rowList=list(row)
    else: 
        if type(row)==type([0]):
            rowList=row
        else: print('setRow error: arg row must be either matrix or list')
    if len(rowList)!=len(mat.row(0)):
        print('setRow error: mat.row() and row have differing lengths')
        return None
    for i in range(len(rowList)):
        mat[rowNum,i]=rowList[i]
        
def set_col(mat,colNum,col):
    colList=list(col)
    if len(colList)!=len(mat.col(0)):
        print('SetCol error: mat.col() and col have differing lengths')
        return None
    for i in range(len(colList)):
        mat[i,colNum]=colList[i]

In [347]:
def nilp_exp(M,step):
    """Computes (exp(M),exp(-M)) for a nilpotent matrix M with M**step=0
    INPUTS:
    * 'M' -- a nilpotent matrix
    * 'step' -- an integer so that M**step=0
    """
    p_cache=[eye(*shape(M))]
    for i in range(1,step):
        p_cache.append(p_cache[-1]*M)
    r1=sum([p_cache[i]/factorial(i) for i in range(1,len(p_cache))],p_cache[0])
    r2=sum([(-1)**i*p_cache[i]/factorial(i) for i in range(1,len(p_cache))],p_cache[0])
    return (r1,r2)

In [18]:
def coords_to_lin_comb(basis,coords):
    '''args: basis, a list of symbols, and coords, a vector of the same length
       Returns: A LinComb corresponding to the vector coords
       NOTE: basis must be a list of symbols'''
    
    # Check if the lengths are the same:
    if len(basis)!=len(coords):
        print('coords_to_lin_comb error: |basis| and |coords| have different lengths')
        return None
    
    result=0
    for i in range(len(basis)):
        result=result+coords[i]*basis[i]
    return result

In [19]:
def coordinatize_cochain_in_basis(c,basis):
    '''c: a cochain object of homogeneous degree
       basis: a collection of str tuples representing elementary cochains
       returns the vector representation of c w.r.t basis as a list'''
    result=[0]*len(basis)
    for key in c.coeff_dict:
        if key in basis: result[basis.index(key)]=c.coeff_dict[key]
        else: print('coordinatize_cochain_in_basis error: cochain component',key,'not in basis')
    return result

In [20]:
def convert_T_symb_elt_to_cochain(se):
    '''se: a T_symb_elt object or a T_symb_basis object
       returns: a degree 0 cochain object corresponding to se'''
    if se==0: return cochain({},se.heis_dim)
    return cochain({(str(T_symb_basis[i]),):se.vec_rep[i] 
                    for i in range(len(T_symb_basis))},se.heis_dim)

In [21]:
def permutation_sign(it_1,it_2):
    '''tuple_1, tuple_2: iterables containing the same elements
       returns: the sign of the permutation taking it_1 to it_2'''
    cnt=0
    for i in range(len(it_1)):
        for j in range(i+1,len(it_1)):
            if it_2.index(it_1[j])<it_2.index(it_1[i]):
                cnt+=1
    return (-1)**cnt

In [22]:
def remove_antisymm_zeros(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector or cochain
       result: coeff_dict, but with keys like (e1,e2,e1) removed'''
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key))!=len(key):
            result_dict.pop(key)
    return result_dict

In [23]:
def remove_antisymm_zeros_cochains(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector or cochain
       result: coeff_dict, but with keys like (e1,e1,e2) removed,
       leaving keys like (e1,e2,e1) representing nontrivial cochains'''
    
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key[0:len(key)-1]))!=len(key)-1:
            result_dict.pop(key)
    return result_dict

In [24]:
def merge_coeff_dicts(dict1,dict2):
    result={}
    for key in set(dict1.keys()).union(set(dict2.keys())):
        coeff=0
        if key in dict1:
            coeff+=dict1[key]
        if key in dict2:
            coeff+=dict2[key]
        result[key]=coeff
    return remove_zeros_from_cd(result)

In [25]:
def remove_zeros_from_cd(coeff_dict):
    '''coeff_dict: a dict with integer values
       returns: a copy of coeff_dict with all keys of value 0 removed'''
    return{A:coeff_dict[A] for A in coeff_dict if coeff_dict[A]!=0}

In [26]:
def str_from_vec(v,b):
    r=[]
    t=''
    for i in range(len(v)):
        if v[i]!=0: r.append([v[i],b[i]])
    if len(r)==0: return('0')
    for i in range(len(r)):
        if r[i][0]==-1: t=t+' - '+str(r[i][1])
        elif r[i][0]!=1: 
            if type(r[i][0])==Add: str_a='('+str(r[i][0])+')'
            else: str_a=str(r[i][0])
            if i==0:
                t=t+str_a+'*'+str(r[i][1])
            else: 
                t=t+' + '+str_a+'*'+str(r[i][1])
        else:
            if i==0:t=str(r[i][1])
            else: t=t+' + '+str(r[i][1])
    return t

def tuple_to_str(v):
    if v==tuple(): return str(v)
    r='('
    for A in v:
        r=r+str(A)+','
    r=r[0:-1]+')'
    return r

def str_from_coeff_dict(coeff_dict):
    '''coeff_dict: a dict with keys that are printable objects
                   or tuples of printable objects and integer values
       returns: a string representing the dict'''
    coeff_dict=remove_zeros_from_cd(coeff_dict)
    if coeff_dict=={}: return '0'
    key_list=list(coeff_dict.keys())
    result=''
    for key in key_list:
        result+=process_key_to_str(coeff_dict,key)
    if result[0:3]==' + ':
        return result[3:len(result)]
    return result[1:len(result)]

def process_key_to_str(coeff_dict,key):
    """adds value*key to the str r unless key is a dict, in which case
    the method is called recursively on the keys of this dict."""
    r=''
    if type(coeff_dict[key])==dict: # if values are dicts, process those
        for k in coeff_dict[key]: 
            r+=process_key_to_str(coeff_dict[key],k)
    else: 
        if type(key)==tuple:
            key_str='('+','.join([str(A) for A in key])+')'
        else: 
            key_str=str(key)
        if coeff_dict[key]==1:
            r+=' + '+key_str
        elif coeff_dict[key]==-1:
            r+=' - '+key_str
        elif coeff_dict[key]!=0:
            if type(coeff_dict[key])==Add:
                coeff_str='('+str(coeff_dict[key])+')'
            else:
                coeff_str=str(coeff_dict[key])
            r+=' + '+coeff_str+'*'+key_str
    return r

In [27]:
def ZV(v):
    """returns true if V is a zero vector, False otherwise""" 
    try:
        for i in v:
            if i!=0: return False
        return True
    except: return False

In [28]:
def iprod_mat(elts):
    '''elts: an iterable of elements with attribute iprod
       returns: the matrix with (ei.iprod(ej)) as its (i,j)-entry'''
    result=zeros(len(elts))            
    for i in range(len(elts)):
        c1=elts[i]
        for j in range(i,len(elts)):
            c2=elts[j]
            val=c1.iprod(c2)
            result[i,j]=val
            result[j,i]=val
    return result

In [29]:
def augment_with_identity(M):
    '''M: a matrix
       returns: matrix (M|Id(n)), where n=shape(M)[0], the number of columns'''
    result=M
    for i in range(shape(M)[0]):
        new_col=[0]*shape(M)[0]
        new_col[i]=1
        new_col=Matrix([[k] for k in new_col])
        result=result.col_insert(shape(M)[1]+i,new_col)
    return result

In [30]:
def Mat_preim_elt(M,v):
    """Returns a vector w so that M*w=v
    INPUTS:
    * 'M' - A matrix representing a finite dimensional linear operator
    * 'v' - a vector in the codomain of M
    """
    # We don't need to solve this entirely, just find a single preim_elt
    A=M.col_insert(shape(M)[1],Matrix(v))
    a=symbols('a0:{num_symb}'.format(num_symb=shape(M)[1]))
    S=solve_linear_system(A,*a)
    if S==None: return None
    b=[ai for ai in a if ai not in S]
    S1={}
    if len(b)!=0:
        S1[b[0]]=1
        for i in range(1,len(b)): S1[b[i]]=0
    return Matrix([a]).transpose().subs(S).subs(S1)

def new_Mat_preim_elt(M,v):
    """Returns a vector w so that M*w=v
    INPUTS:
    * 'M' - A matrix representing a finite dimensional linear operator
    * 'v' - a vector in the codomain of M
    """
    # We don't need to solve this entirely, just find a single preim_elt
    A,cols=M.col_insert(shape(M)[1],Matrix(v)).rref()
    r=zeros(shape(M)[1],1)
    for j in cols:
        i=0
        while A[i,j]==0: i+=1
        r[j]=A[i,-1]
    return r

def col_sp_and_preim(M):
    '''arg: a Matrix M representing a finite dimensional linear operator
       returns: a pair of lists of coords (A,B) where
                A = elements of the domain which map to the elts of B
                B = basis for the columnspace of M'''
    result=([],[])
    
    # transpose and augment
    T=transpose(M)
    T=augment_with_identity(T)
    ## row reduce
    red=T.rref()
    T=red[0]

    piv=red[1]
    i=0
    while piv[i]<shape(M)[0]:
        result[0].append(T.row(i)[shape(M)[0]:len(T.row(i))])
        result[1].append(T.row(i)[0:shape(M)[0]])
        i+=1
        if i==len(piv): break
    return result

def augment_with_identity(M):
    '''M: a matrix
       returns: matrix (M|Id(n)), where n=shape(M)[0], the number of columns'''
    result=M
    for i in range(shape(M)[0]):
        new_col=[0]*shape(M)[0]
        new_col[i]=1
        new_col=Matrix([[k] for k in new_col])
        result=result.col_insert(shape(M)[1]+i,new_col)
    return result

In [31]:
def perm_sign(L,sort_L):
    '''args: two lists with the same elements;
       returns: the sign of the permutation (slow)'''
    result=1
    for j in range(len(L)):
        pj=sort_L.index(L[j])
        for i in range(j):
            if pj<sort_L.index(L[i]): result=result*(-1)
    return result

In [32]:
def mut_mat_copy(A):
    r=zeros(*shape(A))
    for i in range(shape(A)[0]):
        for j in range(shape(A)[1]):
            r[i,j]=A[i,j]
    return r

In [33]:
def ind_der(ind_expr,i):
    '''args: ind_expr, an expression in coordinates h,e,y, and indexed objects, and a natural number i
       returns: the normal form of the derivative of the expression in the i direction'''
    # I'm not sure if this simplification will help or hurt time efficiency
    print('type:',type(ind_expr))
    if type(ind_expr) in [Matrix, ImmutableDenseMatrix]:
        if type(ind_expr)==ImmutableDenseMatrix: result = mut_mat_copy(ind_expr)
        else: result = copy.copy(ind_expr)
        for i in range(shape(result)[0]):
            for j in range(shape(result)[1]):
                result[i,j]=ind_der(result[i,j])
        return result
    
    ind_expr=simplify(ind_expr)
    if isinstance(ind_expr,numbers.Number):
        return 0
    if type(ind_expr)==Add:
        result = Add(*[abn_ind_der(A,i) for A in ind_expr.args])
        return normal_form(result)
    if type(ind_expr)==Mul:
        result=0
        for j in range(len(ind_expr.args)):
            result+=abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
        return normal_form(result)
    # Here I assume the exponents are constant
    if type(ind_expr)==Pow:
        return normal_form(ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*abn_ind_der(ind_expr.base,i))
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        ind=list(ind_expr.indices)
        return normal_form(base[ind+[i]])
    if type(ind_expr)==Symbol:
        if ind_expr in [y,h,e] and [y,h,e].index(ind_expr)==i: return 1
        else: return 0
    
def abn_ind_der(self,ind_expr,i):
    """Returns the derivative of ind_expr in the direction X_i among X_3, X_4,..., X_{2n-1},
    which is a frame on the base manifold.
    
    Note: This only works for horizontal derivatives!
    
    INPUTS:
    * 'ind_expr' -- an expression in h, e, y, and indexed objects
    * 'i' -- an integer between 3 and 2n-1
    """
    if type(ind_expr) in [Matrix, ImmutableDenseMatrix, MutableSparseMatrix, ImmutableSparseMatrix]:
        if type(ind_expr)==ImmutableDenseMatrix: result = mut_mat_copy(ind_expr)
        else: result = copy.copy(int_expr)
        for j in range(shape(result)[0]):
            for k in range(shape(result)[1]):
                result[j,k]=abn_ind_der(result[j,k],i)
        return result

    # # I'm not sure if this simplification will help or hurt time efficiency
    # ind_expr=simplify(ind_expr)
    if isinstance(ind_expr,numbers.Number):
        return 0
    if type(ind_expr)==Add:
        result = Add(*[abn_ind_der(A,i) for A in ind_expr.args])
        return result
    if type(ind_expr)==Mul:
        result=0
        for j in range(len(ind_expr.args)):
            result+=abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
        return result
    if type(ind_expr)==Pow:
        return ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*abn_ind_der(ind_expr.base,i)
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        ind=list(ind_expr.indices)
        return base[ind+[i]]
    if type(ind_expr)==Symbol: return 0
    if type(ind_expr)==exp:
        return ind_expr*abn_ind_der(ind_expr.args[0],i)

In [34]:
def reduce_numer(expr):
    '''arg: expr, a rational expression in symbolic variables 
       returns: the simplest form of the numerator of expr'''
    return numer(simplify(ratsimp(expr)))

In [35]:
def Indexed_factors(expr):
    '''args: expr, a polynomial in Indexed objects
       returns: a set of the Indexed objects involved in expr'''
    if type(expr)==Pow:
        return Indexed_factors(expr.as_base_exp()[0])
    if type(expr)==Mul:
        return set.union(*[Indexed_factors(A) for A in expr.as_coeff_mul()[1]])
    if type(expr)==Add:
        return set.union(*[Indexed_factors(A) for A in expr.as_coeff_add()[1]])
    if type(expr)==Indexed:
        return set([expr])
    return set()

The below method takes an $k-$upper diagonal matrix $A$ and returns $A\wedge A$. Upper triangular means that $j<i\Rightarrow A_j^i = 0$. For $i_1<i_2,j_1<j_2$, we have $(A\wedge A)_{j_1j_2}^{i_1i_2} = A_{j_1}^{i_1}A_{j_2}^{i_2}-A_{j_2}^{i_1}A_{j_1}^{i_2}
$, so we have cases

$$ (A\wedge A)_{j_1j_2}^{i_1i_2} = 
\begin{cases}
    0 \quad\text{if}\ j_1<i_1\ \text{or}\ j_2<i_2
    \\
    A_{j_1}^{i_1}A_{j_2}^{i_2}\quad\text{if}\ i_1\leq j_1 < i_2\leq j_2
    \\
    A_{j_1}^{i_1}A_{j_2}^{i_2}-A_{j_2}^{i_1}A_{j_1}^{i_2}\quad\text{if}\ i_1<i_2\leq j_1<j_2
\end{cases}
$$

$k$-upper triangular means that $j<i+k\Rightarrow A_j^i = 0$, and we again have cases

$$ (A\wedge A)_{j_1j_2}^{i_1i_2} = 
\begin{cases}
    0 \quad\text{if}\ j_1<i_1+k\ \text{or}\ j_2<i_2+k
    \\
    A_{j_1}^{i_1}A_{j_2}^{i_2}\quad\text{if}\ i_1+k\leq j_1 < i_2+k\leq j_2
    \\
    A_{j_1}^{i_1}A_{j_2}^{i_2}-A_{j_2}^{i_1}A_{j_1}^{i_2}\quad\text{if}\ i_1+k<i_2+k\leq j_1<j_2
\end{cases}
$$

It may be useful to consider an even more specific kind of matrix. All our frame change matrices have the form ???

In [38]:
def wedge_Mat(A,k):
    '''arg: A, a k-upper triangular matrix (k UT ==> k-1 UT, and UT <==> 0 UT)
       result: A\\wedge A as a matrix with lex r_ord on the basis
       (i.e., (1,2)<(1,3)<(1,4)<(2,3)<(2,4)<(3,4))'''
    n,m=shape(A) # n=#rows, m=#cols
    
    # Check that A is k-UT
    if not kUT_test(A,k): raise wedge_Mat_Exception(A,'Given matrix must be k-UT: ')
    
    # For fast index conversion, keep track of indices in a dict
    r_ord={}
    ctr=0
    for i in range(n):
        for j in range(i+1,n):
            r_ord[(i,j)]=ctr
            ctr+=1
            
    c_ord={}
    ctr=0
    for i in range(m):
        for j in range(i+1,m):
            c_ord[(i,j)]=ctr
            ctr+=1
    
    result=SparseMatrix(zeros(len(list(r_ord.keys())),len(list(c_ord.keys()))))
    
    # i1+k<=j1<i2+k<=j2
    for i1 in range(n):
        for j1 in range(max(i1+k,0),m):
            for i2 in range(j1-k+1,n):
                for j2 in range(i2+k,m):
                    result[r_ord[(i1,i2)],c_ord[(j1,j2)]] = A[i1,j1]*A[i2,j2]
    
    # i1+k<i2+k<=j1<j2
    for i1 in range(n):
        for i2 in range(i1+1,n):
            for j1 in range(max(i2+k,0),m):
                for j2 in range(j1+1,m):
                    result[r_ord[(i1,i2)],c_ord[(j1,j2)]] = A[i1,j1]*A[i2,j2]-A[i2,j1]*A[i1,j2]
    return result

In [39]:
def kUT_test(A,k):
    '''Tests if a matrix A is k-upper triangular'''
    for i in range(shape(A)[0]):
        for j in range(0,min(i+k,shape(A)[1])):
            if A[i,j]!=0: return False
    return True

In [40]:
class wedge_Mat_Exception(Exception):
     def __init__(self, A, message="wedge_Mat failure:"):
        self.message = message + str(A)
        super().__init__(self.message)

In [41]:
# ## Minitest of wedge_Mat
# from sympy import *
# A1=Matrix([[1,0,-2,-3],[0,1,3,0],[0,0,2,1],[0,0,0,0]])
# test1=Matrix([[1,3,0,2,3,9],[0,2,1,0,0,4],[0,0,0,0,0,0],[0,0,0,2,1,3],[0,0,0,0,0,0],[0,0,0,0,0,0]])
# print(wedge_Mat(A1,0)==test1)

# A2=Matrix([[0,1,4],[1,0,2],[0,2,3],[0,0,-1]])
# test2=Matrix([[-1,-4,2],[0,0,-5],[0,0,-1],[2,3,-4],[0,-1,0],[0,0,-2]])
# print(wedge_Mat(A2,-1)==test2)

# A3=Matrix([[0,2,1,3],[0,0,-1,0],[0,0,0,2],[0,0,0,0],[0,0,0,0]])

# test3=zeros(10,6)
# test3[0,3]=-2
# test3[1,4]=4
# test3[0,5]=3
# test3[1,5]=2
# test3[4,5]=-2
# print(wedge_Mat(A3,1)==test3)

### reduce_numer test

In [43]:
# K=IndexedBase('K')

In [44]:
# P1=((K[0]*K[1])/(K[2]+K[3])+K[1]**2-7*K[2])/(K[8]+K[2])-(4*K[3]/5*K[1]+(K[4]**3-K[1]*K[2])/(K[5]**4+1)/(
#     -3*K[2]**2+K[1])-K[3]/(K[5]+K[1]/K[6]))/(K[4]**2-7*K[1]+3/K[4])


In [45]:
# display(P1)

In [46]:
# reduce_numer(P1)

Each expression involving the structure functions $K$ can be written in many ways because of the Jacobi identity. We can write such expressions uniquely by requiring the lower indices to be ordered as well as the derivatives. Writing $K_{i,j,s}^k$ for $\mathcal{L}_sK^k_{i,j}$, we can apply commutation law

$$\mathcal{L}_s\mathcal{L}_tK = \mathcal{L}_{[s,t]}K + \mathcal{L}_t\mathcal{L}_sK = K_{s,t}^r\mathcal{L}_{r}K + \mathcal{L}_t\mathcal{L}_sK$$

to order derivatives and antisymmetry to order the lower indices.

In [48]:
NF_dict={}

In [49]:
# Note: This uses reference to m, which isn't defined until Geometric Tanaka Prolongations is run

def normal_form(ind_expr,m):
    '''arg: ind_expr, a polynomial of indexed objects which each 
            represent structure functions for the symplectified distribution
            (ex: K[1,3,3,8,9] is the 8,9 derivative of 2-cochain K[1,3,3])
       returns: The normal form of the expression, exchanging derivatives as needed'''
    if ind_expr in NF_dict:
        return NF_dict[ind_expr]
    if type(ind_expr)==Add:
        result = simplify(Add(*[normal_form(A,m) for A in ind_expr.args]))
        NF_dict[ind_expr] = result
        return result
    if type(ind_expr)==Mul:
        result=simplify(Mul(*[normal_form(A,m) for A in ind_expr.args]))
        NF_dict[ind_expr] = result
        return result
    if isinstance(ind_expr,numbers.Number):
        NF_dict[ind_expr]=ind_expr
        return ind_expr
    if type(ind_expr)==Symbol:
        return ind_expr
    if type(ind_expr)==Pow:
        return normal_form(ind_expr.base,m)**normal_form(ind_expr.exp,m)
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        deg=2
        im_ind=ind_expr.indices[2:3]
        ind=ind_expr.indices[0:2]
        ders=ind_expr.indices[3:len(ind_expr.indices)]
        
        if list(ind)!=sorted(ind):
            new_ind=sorted(ind)
            sgn=perm_sign(ind,new_ind)
            L=list(new_ind)+list(im_ind)+list(ders)
            result=simplify(normal_form(sgn*base[L],m))
            NF_dict[ind_expr]=result
            return result
        
        # Only 3 indices, in correct ordering
        if len(ind_expr.indices)==3:
            NF_dict[ind_expr]=ind_expr
            return ind_expr
        
        # More indices --> derivatives to check
        if list(ders)==sorted(ders):
            NF_dict[ind_expr]=ind_expr
            return ind_expr
        j=0 # Least int so that der[j]>der[j+1]
        while ders[j]<=ders[j+1]:
            j+=1
        
        result=base[list(im_ind)+list(ind)+list(ders[0:j])+[ders[j+1],ders[j]]]
        for l in range(2*m+5):
            result=result+base[ders[j],ders[j+1],l]*base[list(ind)+list(im_ind)+list(ders[0:j])+[l]]
        for i in ders[j+2:len(ders)]:
            result=abn_ind_der(result,i)
        result=normal_form(result,m)
        NF_dict[ind_expr]=result
        return result

In [50]:
# from sympy import *
# # An ad hoc test for col_sp_and_preim
# for m in range(1,10):
#     for n in range(1,10):
#         for qqqq in range(1,10):
#             A=randMatrix(m,n)
#             result=col_sp_and_preim(A)
#             r0=result[0]
#             r1=result[1]
#             for i in range(len(r0)):
#                 v= Matrix([[k] for k in r0[i]])
#                 test1=A*v
#                 if tuple(test1)!=r1[i]:
#                     print('Failure for A =', A)

In [51]:
# def coordinatize(b_mat,vec):
#     '''args: b_mat, a matrix such that the span of its columns include vec, a list
#        returns: a list coorinatizing vec in the basis b_mat'''
    

In [52]:
import itertools
def s(n):
    comb_list=itertools.combinations(range(0,len(a)), n)
    return sum([prod([a[i] for i in comb]) for comb in comb_list])

In [53]:
def wghted_mul(M1,w1,M2,w2,m):
    '''args: M and N, a (2m+5)x(2m+5) matrix of degree w1 and w2, respectively
       returns: MN'''
    f=[0,1,3]+list(range(4,2*m+6)) # f[i]:f[i+1] gives the ith piece
    r=zeros(2*m+5)
    if w1+w2>2*m+4: return r
    for i in range(2*m+4-w1-w2):
        j=i+w1+w2
        M1_s = M1[f[i]:f[i+1],f[i+w1]:f[i+w1+1]]
        M2_s = M2[f[i+w1]:f[i+w1+1],f[j]:f[j+1]]
        r[f[i]:f[i+1],f[j]:f[j+1]] = M1_s*M2_s
    return r

In [54]:
def homog_exp(M,w,m):
    '''args: M, a square matrix of size 2m+5
             w, the graded weight of M
             m, such that len(T.basis) = 2m+5
       returns: exp(M)'''
    if w==0:
        r=diag(*[exp(M[i,i]) for i in range(shape(M)[0])])
        r[1:3,1:3]=exp(M[1:3,1:3])
    if w>0:
        s=(2*m+2)//w
        L=[eye(2*m+5),M]
        for i in range(1,s+1):
            L.append(wghted_mul(M,w,L[i],i*w,m))
        r=sum([L[k]/factorial(k) for k in range(len(L))],start=zeros(2*m+5))
    return r

In [55]:
# # # TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST 
# # # wghted_mul test
# # # homog_exp test
# from sympy import *

# m=2
# M1=eye(9)
# w1=0

# M2=zeros(9)
# for i in range(8):
#     M2[i,i+1]=1
# M2[1,2]=0
# M2[1,3]=1
# M2[0,2]=1
# w2=1

# M3=zeros(9)
# for i in range(8):
#     M3[i,i+1]=i+2-i**2
# M3[1,2]=0
# M3[1,3]=4
# M3[0,2]=-2
# w3=1

# M4=zeros(9)
# M4[0,6]=-9
# M4[1,7]=2
# M4[2,7]=3
# M4[3,8]=-55
# w4=5

# t11=M1*M1==wghted_mul(M1,w1,M1,w1,m)
# t12=M1*M2==wghted_mul(M1,w1,M2,w2,m)
# t13=M1*M3==wghted_mul(M1,w1,M3,w3,m)
# t14=M1*M4==wghted_mul(M1,w1,M4,w4,m)
# t22=M2*M2==wghted_mul(M2,w2,M2,w2,m)
# t23=M2*M3==wghted_mul(M2,w2,M3,w3,m)
# t24=M2*M4==wghted_mul(M2,w2,M4,w4,m)
# t33=M3*M3==wghted_mul(M3,w3,M3,w3,m)
# t34=M3*M4==wghted_mul(M3,w3,M4,w4,m)
# t44=M4*M4==wghted_mul(M4,w4,M4,w4,m)

# wm_tests=[t11,t12,t13,t14,t22,t23,t24,t33,t34,t44]
# for i in range(len(wm_tests)):
#     if not wm_tests[i]: print('wghted_mul failure at ',i)
        
# t1=exp(M1)==homog_exp(M1,w1,m)
# t2=exp(M2)==homog_exp(M2,w2,m)
# t3=exp(M3)==homog_exp(M3,w3,m)
# t4=exp(M4)==homog_exp(M4,w4,m)

# he_tests=[t1,t2,t3,t4]
# for i in range(len(he_tests)):
#     if not he_tests[i]: print('homog_exp failure at ', i)

In [56]:
def rand_UT(n,Min=0,Max=99,nonzero_entries=None):
    R=eye(n)
    if nonzero_entries==None:
        R=randMatrix(n,min=Min,max=Max)
        for i in range(n):
            R[i,i]=1
            for j in range(i):
                R[i,j]=0
        return R           
    
    R=eye(n)
    ctr=0
    while ctr<=nonzero_entries:
        i=random.randint(0,n-2)
        j=random.randint(i+1,n-1)
        R[i,j]=random.randint(Min,Max)
        ctr+=1
    return R